# **Deber 4**
Martín Montero<br>
00328595

## **1. Configuraciones**

### **Entorno para Decrifrado TLS**
Para que Wireshark pueda "ver" dentro del tráfico HTTPS, se configura Python para que guarde las llaves de cifrado en un archivo local.

In [ ]:
import os
import requests

os.environ['SSLKEYLOGFILE'] = 'sslkeys.log'

print(f"Archivo de llaves configurado en: {os.path.abspath('sslkeys.log')}")

BASE_URL = "https://jsonplaceholder.typicode.com"

Archivo de llaves configurado en: /home/martin/Desktop/Main/USFQ/Semestre_8/Redes/Teoria/Deber 4/sslkeys.log


Se agrega el archivo con las llaves de cifrado a Wireshark, en Edit > Preferences > Protocols > TLS.

<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/tls_wireshark.png" width="60%">

### **Headers para Filtrado**
Se crea una etiqueta personalizada para filtrar fácilmente en Wireshark

In [ ]:
HEADERS_BASE = {
    'X-Redes': 'Deber-4-USFQ', # Se puede filtrar con "http contains "Deber-4-USFQ"
    'Content-Type': 'application/json'
}

## **2. Desarrollo de los Enunciados**

### **a. Recuperar todos los usuarios**
Se realiza una petición `GET` al endpoint `/users`. Se itera sobre la lista para extraer el `id` y el `name`.

In [26]:
def get_all_users():
    print("--- Lista de Usuarios ---")
    response = requests.get(f"{BASE_URL}/users", headers=HEADERS_BASE)
    
    if response.status_code == 200:
        users = response.json()
        for user in users:
            print(f"ID: {user['id']} | Nombre: {user['name']}")
    else:
        print(f"Error en la consulta: {response.status_code}")

get_all_users()

--- Lista de Usuarios ---
ID: 1 | Nombre: Leanne Graham
ID: 2 | Nombre: Ervin Howell
ID: 3 | Nombre: Clementine Bauch
ID: 4 | Nombre: Patricia Lebsack
ID: 5 | Nombre: Chelsey Dietrich
ID: 6 | Nombre: Mrs. Dennis Schulist
ID: 7 | Nombre: Kurtis Weissnat
ID: 8 | Nombre: Nicholas Runolfsdottir V
ID: 9 | Nombre: Glenna Reichert
ID: 10 | Nombre: Clementina DuBuque


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `50168` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
> Se observa el método de consulta y la etiqueta personalizada para el filtrado.
```http
    GET /users HTTP/1.1\r\n
    Host: jsonplaceholder.typicode.com\r\n
    User-Agent: python-requests/2.33.1\r\n
    Accept-Encoding: gzip, deflate, zstd\r\n
    Accept: */*\r\n
    Connection: keep-alive\r\n
    X-Redes: Deber-4-USFQ\r\n
    Content-Type: application/json\r\n
```

**Respuesta del Servidor (Response Header):**
> El servidor confirma la recepción exitosa y el envío de datos en formato JSON.
```http
    HTTP/1.1 200 OK\r\n
    Date: Wed, 22 Apr 2026 04:19:18 GMT\r\n
    Content-Type: application/json; charset=utf-8\r\n
    Content-Length: 1847\r\n
    Connection: keep-alive\r\n
    access-control-allow-credentials: true\r\n
    Cache-Control: max-age=43200\r\n
    Content-Encoding: gzip\r\n
    etag: W/"160d-1eMSsxeJRfnVLRBmYJSbCiJZ1qQ"\r\n
    expires: -1\r\n
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}\r\n
    pragma: no-cache\r\n
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=DLNAOqUhMrbAscoEX6HKrEYuLRPBENjfSCAgKnw0Hrw%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1773308320"}],"max_age":3600}\r\n
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=DLNAOqUhMrbAscoEX6HKrEYuLRPBENjfSCAgKnw0Hrw%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1773308320"\r\n
    Server: cloudflare\r\n
    vary: Origin, Accept-Encoding\r\n
    via: 2.0 heroku-router\r\n
    x-content-type-options: nosniff\r\n
    x-powered-by: Express\r\n
    x-ratelimit-limit: 1000\r\n
    x-ratelimit-remaining: 998\r\n
    x-ratelimit-reset: 1773308335\r\n
    Age: 1169\r\n
    Accept-Ranges: bytes\r\n
    cf-cache-status: HIT\r\n
    CF-RAY: 9f01d459197ca9db-ATL\r\n
    alt-svc: h3=":443"; ma=86400\r\n
```

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response_payload.png" width="80%">

### **b. Recuperar posts de un usuario particular**
Se utiliza un parámetro de consulta (`params`) para filtrar los posts por `userId`.

In [27]:
def get_posts_by_user(user_id):
    print(f"\n--- Posts de Usuario {user_id} ---")
    payload = {'userId': user_id}
    response = requests.get(f"{BASE_URL}/posts", params=payload, headers=HEADERS_BASE)
    
    if response.status_code == 200:
        posts = response.json()
        for post in posts:
            print(f"Post ID: {post['id']} | Título: {post['title']}")
    else:
        print(f"Error: {response.status_code}")

get_posts_by_user(1)


--- Posts de Usuario 1 ---
Post ID: 1 | Título: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
Post ID: 2 | Título: qui est esse
Post ID: 3 | Título: ea molestias quasi exercitationem repellat qui ipsa sit aut
Post ID: 4 | Título: eum et est occaecati
Post ID: 5 | Título: nesciunt quas odio
Post ID: 6 | Título: dolorem eum magni eos aperiam quia
Post ID: 7 | Título: magnam facilis autem
Post ID: 8 | Título: dolorem dolore est ipsam
Post ID: 9 | Título: nesciunt iure omnis dolorem tempora et accusantium
Post ID: 10 | Título: optio molestias id quia eum


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `50168` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
> Se observa el método de consulta y la etiqueta personalizada para el filtrado.
```http
    GET /posts?userId=1 HTTP/1.1\r\n
    Host: jsonplaceholder.typicode.com\r\n
    User-Agent: python-requests/2.33.1\r\n
    Accept-Encoding: gzip, deflate, zstd\r\n
    Accept: */*\r\n
    Connection: keep-alive\r\n
    X-Redes: Deber-4-USFQ\r\n
    Content-Type: application/json\r\n
```

**Respuesta del Servidor (Response Header):**
> El servidor confirma la recepción exitosa y el envío de datos en formato JSON.
```http
    HTTP/1.1 200 OK\r\n
    Date: Wed, 22 Apr 2026 04:58:31 GMT\r\n
    Content-Type: application/json; charset=utf-8\r\n
    Content-Length: 1002\r\n
    Connection: keep-alive\r\n
    access-control-allow-credentials: true\r\n
    Cache-Control: max-age=43200\r\n
    Content-Encoding: gzip\r\n
    etag: W/"aa6-j2NSH739l9uq40OywFMn7Y0C/iY"\r\n
    expires: -1\r\n
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}\r\n
    pragma: no-cache\r\n
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=8i%2FId1g%2Fhu78sWavABWZA1OoCvVStDGj5qVEgBAtG9g%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1774888683"}],"max_age":3600}\r\n
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=8i%2FId1g%2Fhu78sWavABWZA1OoCvVStDGj5qVEgBAtG9g%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1774888683"\r\n
    Server: cloudflare\r\n
    vary: Origin, Accept-Encoding\r\n
    via: 2.0 heroku-router\r\n
    x-content-type-options: nosniff\r\n
    x-powered-by: Express\r\n
    x-ratelimit-limit: 1000\r\n
    x-ratelimit-remaining: 999\r\n
    x-ratelimit-reset: 1774888733\r\n
    Age: 12402\r\n
    Accept-Ranges: bytes\r\n
    cf-cache-status: HIT\r\n
    CF-RAY: 9f020dcb0ddbe155-ATL\r\n
    alt-svc: h3=":443"; ma=86400\r\n
```

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_response_payload.png" width="80%">

### **c. Crear un nuevo post**
Se utiliza el método `POST`. Enviamos un objeto JSON en el cuerpo de la petición.

In [28]:
def create_post():
    print("\n--- Nuevo Post ---")
    new_post = {
        'title': 'Redes USFQ',
        'body': 'Analizando tráfico con Wireshark',
        'userId': 1
    }
    
    response = requests.post(f"{BASE_URL}/posts", json=new_post, headers=HEADERS_BASE)
    
    if response.status_code == 201:
        print("Post creado exitosamente (Simulado).")
        print(f"Respuesta del servidor: {response.json()}")
    else:
        print(f"Error: {response.status_code}")

create_post()


--- Nuevo Post ---
Post creado exitosamente (Simulado).
Respuesta del servidor: {'title': 'Redes USFQ', 'body': 'Analizando tráfico con Wireshark', 'userId': 1, 'id': 101}


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `50168` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
> Se observa el método de consulta y la etiqueta personalizada para el filtrado.
```http
    POST /posts HTTP/1.1\r\n
    Host: jsonplaceholder.typicode.com\r\n
    User-Agent: python-requests/2.33.1\r\n
    Accept-Encoding: gzip, deflate, zstd\r\n
    Accept: */*\r\n
    Connection: keep-alive\r\n
    X-Redes: Deber-4-USFQ\r\n
    Content-Type: application/json\r\n
    Content-Length: 85\r\n
```

**Cuerpo de la Petición (Payload):**  
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_request_payload.png" width="80%">

**Respuesta del Servidor (Response Header):**
> El servidor confirma la recepción exitosa y el envío de datos en formato JSON.
```http
    HTTP/1.1 201 Created\r\n
    Date: Wed, 22 Apr 2026 05:02:51 GMT\r\n
    Content-Type: application/json; charset=utf-8\r\n
    Content-Length: 102\r\n
    Connection: keep-alive\r\n
    access-control-allow-credentials: true\r\n
    access-control-expose-headers: Location\r\n
    Cache-Control: no-cache\r\n
    etag: W/"66-Y7RrWqoAP9PzGe2g4FPi/evlWpE"\r\n
    expires: -1\r\n
    location: https://jsonplaceholder.typicode.com/posts/101\r\n
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}\r\n
    pragma: no-cache\r\n
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=wm0lId9bp6z9txrV1ItDHppJWKwDmi5wQ1CCOqwPMSQ%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776834171"}],"max_age":3600}\r\n
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=wm0lId9bp6z9txrV1ItDHppJWKwDmi5wQ1CCOqwPMSQ%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776834171"\r\n
    Server: cloudflare\r\n
    vary: Origin, X-HTTP-Method-Override, Accept-Encoding\r\n
    via: 2.0 heroku-router\r\n
    x-content-type-options: nosniff\r\n
    x-powered-by: Express\r\n
    x-ratelimit-limit: 1000\r\n
    x-ratelimit-remaining: 999\r\n
    x-ratelimit-reset: 1776834216\r\n
    cf-cache-status: DYNAMIC\r\n
    CF-RAY: 9f0214246a21dd21-ATL\r\n
    alt-svc: h3=":443"; ma=86400\r\n
```

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_response_payload.png" width="80%">

### **d. Actualizar un determinado post**
Se utiliza el método `PUT` para actualizar el recurso completo en el ID especificado.

In [29]:
def update_post(post_id):
    print(f"\n--- Actualizar Post {post_id} ---")
    updated_data = {
        'id': post_id,
        'title': 'Título Actualizado',
        'body': 'Contenido modificado para el deber 4',
        'userId': 1
    }
    
    response = requests.put(f"{BASE_URL}/posts/{post_id}", json=updated_data, headers=HEADERS_BASE)
    
    if response.status_code == 200:
        print("Post actualizado con éxito.")
        print(f"Datos recibidos: {response.json()}")
    else:
        print(f"Error: {response.status_code}")

update_post(1)


--- Actualizar Post 1 ---
Post actualizado con éxito.
Datos recibidos: {'id': 1, 'title': 'Título Actualizado', 'body': 'Contenido modificado para el deber 4', 'userId': 1}


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `50168` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
> Se observa el método de consulta y la etiqueta personalizada para el filtrado.
```http
    PUT /posts/1 HTTP/1.1\r\n
    Host: jsonplaceholder.typicode.com\r\n
    User-Agent: python-requests/2.33.1\r\n
    Accept-Encoding: gzip, deflate, zstd\r\n
    Accept: */*\r\n
    Connection: keep-alive\r\n
    X-Redes: Deber-4-USFQ\r\n
    Content-Type: application/json\r\n
    Content-Length: 106\r\n
```

**Cuerpo de la Petición (Payload):**  
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_request_payload.png" width="80%">

**Respuesta del Servidor (Response Header):**
> El servidor confirma la recepción exitosa y el envío de datos en formato JSON.
```http
    HTTP/1.1 200 OK\r\n
    Date: Wed, 22 Apr 2026 05:17:31 GMT\r\n
    Content-Type: application/json; charset=utf-8\r\n
    Transfer-Encoding: chunked\r\n
    Connection: keep-alive\r\n
    access-control-allow-credentials: true\r\n
    Cache-Control: no-cache\r\n
    etag: W/"70-LJhQSmDa7+MNkDby1xMHoFfmlIE"\r\n
    expires: -1\r\n
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}\r\n
    pragma: no-cache\r\n
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=sXGgjsqG4LSzCwA1nkqh4oY7UYvtBIwXtQKbemvo14Q%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776835051"}],"max_age":3600}\r\n
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=sXGgjsqG4LSzCwA1nkqh4oY7UYvtBIwXtQKbemvo14Q%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776835051"\r\n
    Server: cloudflare\r\n
    vary: Origin, Accept-Encoding\r\n
    via: 2.0 heroku-router\r\n
    x-content-type-options: nosniff\r\n
    x-powered-by: Express\r\n
    x-ratelimit-limit: 1000\r\n
    x-ratelimit-remaining: 999\r\n
    x-ratelimit-reset: 1776835056\r\n
    cf-cache-status: DYNAMIC\r\n
    Content-Encoding: zstd\r\n
    CF-RAY: 9f02299bfe6f9afd-GYE\r\n
    alt-svc: h3=":443"; ma=86400\r\n
```

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_response_payload.png" width="80%">

### **e. Eliminar un determinado post**
Se utiliza el método `DELETE`.

In [30]:
def delete_post(post_id):
    print(f"\n--- liminar Post {post_id} ---")
    response = requests.delete(f"{BASE_URL}/posts/{post_id}", headers=HEADERS_BASE)
    
    if response.status_code == 200:
        print(f"El post {post_id} ha sido eliminado correctamente.")
    else:
        print(f"Error: {response.status_code}")

delete_post(1)


--- liminar Post 1 ---
El post 1 ha sido eliminado correctamente.


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `50168` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
> Se observa el método de consulta y la etiqueta personalizada para el filtrado.
```http
    DELETE /posts/1 HTTP/1.1\r\n
    Host: jsonplaceholder.typicode.com\r\n
    User-Agent: python-requests/2.33.1\r\n
    Accept-Encoding: gzip, deflate, zstd\r\n
    Accept: */*\r\n
    Connection: keep-alive\r\n
    X-Redes: Deber-4-USFQ\r\n
    Content-Type: application/json\r\n
    Content-Length: 0\r\n
```

**Respuesta del Servidor (Response Header):**
> El servidor confirma la recepción exitosa y el envío de datos en formato JSON.
```http
    HTTP/1.1 200 OK\r\n
    Date: Wed, 22 Apr 2026 05:25:04 GMT\r\n
    Content-Type: application/json; charset=utf-8\r\n
    Content-Length: 2\r\n
    Connection: keep-alive\r\n
    access-control-allow-credentials: true\r\n
    Cache-Control: no-cache\r\n
    etag: W/"2-vyGp6PvFo4RvsFtPoIWeCReyIC8"\r\n
    expires: -1\r\n
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}\r\n
    pragma: no-cache\r\n
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=NoQU1S4Z83L05kqkS%2Fl82%2BXW5OPAt6lf5TVfYCpP6j0%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776835503"}],"max_age":3600}\r\n
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=NoQU1S4Z83L05kqkS%2Fl82%2BXW5OPAt6lf5TVfYCpP6j0%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776835503"\r\n
    Server: cloudflare\r\n
    vary: Origin, Accept-Encoding\r\n
    via: 2.0 heroku-router\r\n
    x-content-type-options: nosniff\r\n
    x-powered-by: Express\r\n
    x-ratelimit-limit: 1000\r\n
    x-ratelimit-remaining: 999\r\n
    x-ratelimit-reset: 1776835536\r\n
    cf-cache-status: DYNAMIC\r\n
    CF-RAY: 9f0234a95efc824d-GYE\r\n
    alt-svc: h3=":443"; ma=86400\r\n
```

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response_payload.png" width="80%">